## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


FAST SHAP positional feature (BATCHED)

In [ ]:
# ==========================================
#  FAST SHAP positional feature (BATCHED)
# Supports: SCIBERT, ROBERTA, BART, T5 (Seq2Seq teacher-forcing),
#           LLAMA3, DEEPSEEK (custom cls head + LoRA + 4bit)
# ==========================================

### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
!pip -q install -U bitsandbytes shap peft transformers

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig,
)
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
# Εισαγωγή του Access Token για LLaMA_3.
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
from huggingface_hub import login

if not os.environ.get("HF_TOKEN"):
    raise ValueError("Set the HF_TOKEN environment variable before loading gated Hugging Face models.")

login(token=os.environ["HF_TOKEN"])

In [ ]:
# ------------------------------
# Global settings
# ------------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 256

# Για ταχύτητα/σταθερότητα στο inference
torch.set_grad_enabled(False)

In [ ]:
# ==============================
#  LOAD DATA (master or final)
# ==============================
BASE_PATH = str(PROCESSED_DATA_DIR)
MASTER_PATH = f"{BASE_PATH}/Master_results_with_features.csv"
FINAL_PATH  = f"{BASE_PATH}/Final_Master_results_with_features.csv"
SAMPLE_PATH = f"{BASE_PATH}/shap_sample_indices.csv"

if os.path.exists(FINAL_PATH):
    print("Loading Final_Master_results_with_features.csv")
    df = pd.read_csv(FINAL_PATH)
else:
    print("Loading Master_results_with_features.csv")
    df = pd.read_csv(MASTER_PATH)

In [ ]:
# ==============================
#  MODEL SELECTION
# ==============================
print("""
Επιλογές:
1 = SCIBERT
2 = ROBERTA
3 = BART
4 = T5
5 = LLAMA3
6 = DEEPSEEK
""")
option = int(input("Επιλογή μοντέλου: ").strip() or "1")

if option == 1:
    CURRENT_MODEL = "SCIBERT"
elif option == 2:
    CURRENT_MODEL = "ROBERTA"
elif option == 3:
    CURRENT_MODEL = "BART"
elif option == 4:
    CURRENT_MODEL = "T5"
elif option == 5:
    CURRENT_MODEL = "LLAMA3"
elif option == 6:
    CURRENT_MODEL = "DEEPSEEK"
else:
    CURRENT_MODEL = "SCIBERT"

print(f"Selected model: {CURRENT_MODEL}")

In [ ]:
# ==============================
# Paths
# ==============================

MODEL_PATHS = {
    "SCIBERT": "external_materials/model_weights/SciBERT/saved_model",
    "ROBERTA": "external_materials/model_weights/RoBERTa/saved_model",
    "BART": "external_materials/model_weights/facebook_bart-large/saved_model",
    "T5": "external_materials/model_weights/t5-large/saved_model",

    "LLAMA3_BASE": "meta-llama/Llama-3.1-8B",
    "LLAMA3_WEIGHTS": "external_materials/model_weights/LLaMA_3/saved_model/saved_llama3_state_dict.pth",

    "DEEPSEEK_BASE": "deepseek-ai/deepseek-llm-7b-base",
    "DEEPSEEK_WEIGHTS": "external_materials/model_weights/DeepSeek/saved_model/saved_deepSeek_state_dict.pth",
}

TOKENIZER_PATHS = {
    "SCIBERT": "external_materials/model_weights/SciBERT/saved_tokenizer",
    "ROBERTA": "external_materials/model_weights/RoBERTa/saved_tokenizer",
    "BART": "external_materials/model_weights/facebook_bart-large/saved_tokenizer",
    "T5": "external_materials/model_weights/t5-large/saved_tokenizer",

    "LLAMA3": "external_materials/model_weights/LLaMA_3/saved_tokenizer",
    "DEEPSEEK": "external_materials/model_weights/DeepSeek/saved_tokenizer",
}


In [ ]:
# ==============================
#  LOAD OR CREATE FIXED SHAP SAMPLE (ONCE)
# ==============================
TARGET_CIT = 2797  # Citations
CONTROL_N = 2864   # non-citations

if not os.path.exists(SAMPLE_PATH):
    print("Creating fixed stratified SHAP sample (ONCE)")

    cit_df = df[df["is_citation"] == 1].copy()
    if len(cit_df) == 0:
        raise ValueError("Δεν βρέθηκαν citations (is_citation==1) στο dataset.")

    # stratified ανά ground_truth (όπως έκανα)
    cit_sample = (
        cit_df
        .groupby("ground_truth", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=min(len(x), int(TARGET_CIT * len(x) / len(cit_df))),
                random_state=SEED
            ),
            include_groups=False
        )
    )

    noncit_df = df[df["is_citation"] == 0]
    control_sample = noncit_df.sample(
        n=min(CONTROL_N, len(noncit_df)),
        random_state=SEED
    )

    shap_indices = cit_sample.index.union(control_sample.index)
    pd.Series(shap_indices).to_csv(SAMPLE_PATH, index=False, header=False)

    print(f"Saved SHAP sample: {len(cit_sample)} citations + {len(control_sample)} controls = {len(shap_indices)} total")
else:
    print("Loading existing fixed SHAP sample")
    shap_indices = pd.read_csv(SAMPLE_PATH, header=None)[0].astype(int).values
    print("Loaded indices:", len(shap_indices))

In [ ]:
# ==============================
#  CUSTOM CLASSES (LLaMA & DeepSeek)
# ==============================
class CustomClassificationModel(nn.Module):
    def __init__(self, base_model, num_labels=3):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]  # (B, T, H)

        mask = attention_mask.unsqueeze(-1).float()  # (B,T,1)
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        logits = self.classifier(pooled)
        return logits

In [ ]:
# ==============================
#  Builders for explainers
# ==============================
def build_encoder_explainer(model_name: str):
    """
    SCIBERT/ROBERTA/BART (as SequenceClassification)
    No pipeline -> better batching & speed.
    """
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATHS[model_name])
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATHS[model_name]).to(DEVICE)
    model.eval()

    def predict(texts):
        inputs = tokenizer(
            list(texts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)

        with torch.inference_mode():
            out = model(**inputs)
            logits = out.logits

        return torch.softmax(logits, dim=-1).detach().cpu().numpy()

    explainer = shap.Explainer(predict, tokenizer)
    return explainer, ""

def build_llm_explainer(model_name: str):
    """
    LLAMA3/DEEPSEEK:
    - load base seq cls model 4bit
    - apply LoRA structure
    - load saved .pth into custom cls head wrapper
    - shap.Explainer(predict, tokenizer)
    """
    base_path = MODEL_PATHS[f"{model_name}_BASE"]
    weights_path = MODEL_PATHS[f"{model_name}_WEIGHTS"]

    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATHS[model_name], fix_mistral_regex=True)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    base_model = AutoModelForSequenceClassification.from_pretrained(
        base_path,
        quantization_config=bnb_config,
        device_map="auto"
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        task_type=TaskType.SEQ_CLS,
        lora_dropout=0.05,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    )
    base_model = get_peft_model(base_model, lora_config)

    model = CustomClassificationModel(base_model, num_labels=3)
    state = torch.load(weights_path, map_location="cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(state, strict=False)
    model = model.to(DEVICE)
    model.eval()

    def predict(texts):
        inputs = tokenizer(
            list(texts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)

        with torch.inference_mode():
            logits = model(inputs["input_ids"], attention_mask=inputs["attention_mask"])

        return torch.softmax(logits, dim=-1).detach().cpu().numpy()

    explainer = shap.Explainer(predict, tokenizer)
    return explainer, ""

def build_t5_explainer():
    """
    T5 (AutoModelForSeq2SeqLM) αλλά *χωρίς generate*.
    Υπολογίζουμε p(class) από log-probabilities των label sequences με teacher forcing.

    Works even if labels are multi-token.
    """
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATHS["T5"])
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATHS["T5"]).to(DEVICE)
    model.eval()

    LABEL_TEXTS = ["negative", "neutral", "positive"]
    # label token ids (no special tokens)
    label_ids_list = [tokenizer.encode(l, add_special_tokens=False) for l in LABEL_TEXTS]
    max_lab_len = max(len(x) for x in label_ids_list)

    # Prepare padded label ids + mask
    label_ids = []
    label_mask = []
    for ids in label_ids_list:
        pad_len = max_lab_len - len(ids)
        label_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
        label_mask.append([1]*len(ids) + [0]*pad_len)

    label_ids = torch.tensor(label_ids, device=DEVICE)            # (3, L)
    label_mask = torch.tensor(label_mask, device=DEVICE).float()  # (3, L)

    def predict(texts):
        # Encode inputs
        enc = tokenizer(
            list(texts),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        ).to(DEVICE)

        B = enc["input_ids"].shape[0]

        # Repeat inputs for 3 labels: (B*3, ...)
        input_ids = enc["input_ids"].repeat_interleave(3, dim=0)
        attn_mask = enc["attention_mask"].repeat_interleave(3, dim=0)

        # Build decoder_input_ids for teacher forcing:
        # decoder_input_ids = [pad] + label_ids[:-1]
        # We pad to max_lab_len as well.
        dec_in = torch.full((B*3, max_lab_len), tokenizer.pad_token_id, device=DEVICE, dtype=torch.long)

        # For each label k, rows corresponding to that label get its ids shifted right
        # rows: k, k+3, k+6, ...
        for k in range(3):
            ids_k = label_ids[k]  # (L,)
            # shift right: [pad] + ids[:-1]
            shifted = torch.cat([torch.tensor([tokenizer.pad_token_id], device=DEVICE), ids_k[:-1]], dim=0)
            dec_in[k::3] = shifted.unsqueeze(0).repeat(B, 1)

        with torch.inference_mode():
            out = model(
                input_ids=input_ids,
                attention_mask=attn_mask,
                decoder_input_ids=dec_in
            )
            # logits: (B*3, L, vocab)
            logits = out.logits

        log_probs = torch.log_softmax(logits, dim=-1)  # (B*3, L, vocab)

        # Gather log-prob for the true label tokens at each position
        # true tokens are label_ids[k] for rows belonging to k
        token_logps = torch.zeros((B*3, max_lab_len), device=DEVICE)

        for k in range(3):
            rows = slice(k, B*3, 3)            # k::3
            true_ids = label_ids[k].unsqueeze(0).repeat(B, 1)  # (B, L)
            gathered = log_probs[rows].gather(2, true_ids.unsqueeze(-1)).squeeze(-1)  # (B,L)
            token_logps[rows] = gathered

        # Mask padding positions (multi-token labels supported)
        # label_mask[k] applied to rows k::3
        masked_sum = torch.zeros((B*3,), device=DEVICE)
        for k in range(3):
            rows = slice(k, B*3, 3)
            m = label_mask[k].unsqueeze(0).repeat(B, 1)  # (B,L)
            masked_sum[rows] = (token_logps[rows] * m).sum(dim=1)  # sum logp(label sequence)

        # Convert sequence log-likelihoods to class probs (softmax over 3 labels)
        ll = masked_sum.view(B, 3)  # (B,3)
        probs = torch.softmax(ll, dim=-1)

        return probs.detach().cpu().numpy()

    explainer = shap.Explainer(predict, tokenizer)
    return explainer, ""

In [ ]:
# ==============================
#  Build explainer
# ==============================
print(f"--- Loading {CURRENT_MODEL} and building SHAP explainer ---")

if CURRENT_MODEL in ["SCIBERT", "ROBERTA", "BART"]:
    explainer, prefix = build_encoder_explainer(CURRENT_MODEL)
elif CURRENT_MODEL == "T5":
    explainer, prefix = build_t5_explainer()
elif CURRENT_MODEL in ["LLAMA3", "DEEPSEEK"]:
    explainer, prefix = build_llm_explainer(CURRENT_MODEL)
else:
    raise ValueError("Unknown model option")

In [ ]:
def compute_shap_pos_batched(
    texts,
    explainer,
    prefix="",
    batch_size=16,
    max_evals=None
):
    """
    Υπολογίζει το positional feature:
    max_idx(|SHAP| across classes) / (num_tokens - 1)

    Επιστρέφει επίσης το keyword token και την θέση του.

    texts: list[str]
    returns: np.array float, shape (N,), list[str], np.array int
    """
    out_pos = np.empty(len(texts), dtype=float)
    out_keyword_token = [None] * len(texts)
    out_max_idx = np.empty(len(texts), dtype=int)

    for start in tqdm(range(0, len(texts), batch_size), desc="SHAP batches"):
        chunk = texts[start:start+batch_size]
        chunk_full = [(prefix + t) if prefix else t for t in chunk]

        try:
            if max_evals is None:
                sv = explainer(chunk_full)
            else:
                # δεν υποστηρίζεται σε όλα τα explainers, οπότε try/except
                sv = explainer(chunk_full, max_evals=max_evals)
        except TypeError:
            sv = explainer(chunk_full)

        for i in range(len(chunk_full)):
            vals = sv.values[i]  # (tokens, classes) συνήθως
            impact = np.abs(vals).max(axis=1)  # (tokens,)

            max_idx = int(np.argmax(impact))
            denom = (len(impact) - 1)
            out_pos[start + i] = (max_idx / denom) if denom > 0 else 0.0
            out_max_idx[start + i] = max_idx

            # Get the keyword token using sv.data (which contains the tokens)
            try:
                # sv.data[i] should be a list or array of tokens (strings)
                keyword_token = sv.data[i][max_idx]
            except Exception:
                keyword_token = "" # Fallback if token extraction fails

            out_keyword_token[start + i] = keyword_token

    return out_pos, out_keyword_token, out_max_idx

In [ ]:
# ==============================
#  Apply to SHAP sample indices
# ==============================
BATCH_SIZE = 8  # δοκιμάζω 16 ή 32 στην A100
MAX_EVALS = 400  # ήταν None  # π.χ. 400 αν θελω έξτρα speed (μικρό tradeoff σε fidelity)

shap_pos_col = f"{CURRENT_MODEL}_shap_pos"
shap_keyword_token_col = f"{CURRENT_MODEL}_shap_keyword_token"
shap_pos_abs_dist_col = f"{CURRENT_MODEL}_shap_pos_abs_dist"
citation_relative_pos_col = "Citation_Relative_Position" # Assuming this column exists in df

# Check if the positional SHAP feature column already exists
if shap_pos_col in df.columns and df[shap_pos_col].notna().any():
    print(f"{shap_pos_col} already exists (has non-NaN values) — skipping SHAP computation.")
else:
    print(f"Computing SHAP positional feature for {CURRENT_MODEL} (batched) ...")
    df[shap_pos_col] = np.nan
    df[shap_keyword_token_col] = None # Initialize new column for keyword token
    df[shap_pos_abs_dist_col] = np.nan # Initialize new column for absolute distance
    # Store the absolute max_idx temporarily to calculate distance, if needed, or for inspection
    temp_max_idx_col = f"{CURRENT_MODEL}_shap_max_idx"
    df[temp_max_idx_col] = np.nan

    sample_texts = df.loc[shap_indices, "sentence"].astype(str).tolist()
    pos_vals, keyword_tokens, max_indices = compute_shap_pos_batched( # Unpack new returns
        texts=sample_texts,
        explainer=explainer,
        prefix=prefix or "",
        batch_size=BATCH_SIZE,
        max_evals=MAX_EVALS
    )

    df.loc[shap_indices, shap_pos_col] = pos_vals
    df.loc[shap_indices, shap_keyword_token_col] = keyword_tokens # Assign new keyword tokens
    df.loc[shap_indices, temp_max_idx_col] = max_indices # Assign max_indices

    # Calculate absolute distance
    if citation_relative_pos_col in df.columns:
        # Assuming Citation_Relative_Position is a normalized value (0-1) like shap_pos_col
        df.loc[shap_indices, shap_pos_abs_dist_col] = np.abs(
            df.loc[shap_indices, shap_pos_col] - df.loc[shap_indices, citation_relative_pos_col]
        )
    else:
        print(f"Warning: Column '{citation_relative_pos_col}' not found in DataFrame. Skipping absolute distance calculation.")

    # Drop the temporary max_idx column after calculation if not needed permanently
    df = df.drop(columns=[temp_max_idx_col])

print("Updated DataFrame head with new columns:")
print(df[[ "sentence", "is_citation", "ground_truth", shap_pos_col, shap_keyword_token_col, shap_pos_abs_dist_col ]].head())

In [ ]:
# ==============================
#  SAVE FINAL CSV
# ==============================
df.to_csv(FINAL_PATH, index=False)
print("Saved:", FINAL_PATH)

shap_pos_col = f"{CURRENT_MODEL}_shap_pos"
shap_keyword_token_col = f"{CURRENT_MODEL}_shap_keyword_token"
shap_pos_abs_dist_col = f"{CURRENT_MODEL}_shap_pos_abs_dist"

df[[ "sentence", "is_citation", "ground_truth", shap_pos_col, shap_keyword_token_col, shap_pos_abs_dist_col ]].head()

END